# 🎸 BOSS IR2 - Test Lettura (RQ1)

Ora che conosciamo il Model ID (**01 05 09**) e gli indirizzi, proviamo a **LEGGERE** i valori dal pedale.

Protocollo RQ1 (Data Request):
```
F0 41 10 01 05 09 11 AA BB CC DD SS TT UU VV ZZ F7
               Model RQ1 Address     Size        Checksum
```

In [ ]:
import mido
import time

# Configurazione
IR2_INPUT = None
IR2_OUTPUT = None

for name in mido.get_input_names():
    if 'BOSS' in name.upper() or 'IR-2' in name.upper():
        IR2_INPUT = name
for name in mido.get_output_names():
    if 'BOSS' in name.upper() or 'IR-2' in name.upper():
        IR2_OUTPUT = name

print(f"In: {IR2_INPUT}")
print(f"Out: {IR2_OUTPUT}")

In [ ]:
def checksum(data):
    return (128 - (sum(data) % 128)) & 0x7F

def send_rq1(addr, size=1):
    # Header BOSS IR-2
    msg = [0x41, 0x10, 0x01, 0x05, 0x09, 0x11]
    
    # Address (4 bytes) + Size (4 bytes)
    # Size 1 byte = 00 00 00 01
    payload = addr + [0, 0, 0, size]
    msg.extend(payload)
    
    # Checksum
    msg.append(checksum(payload))
    
    print(f"📤 RQ1: {' '.join(f'{b:02X}' for b in msg)}")
    
    outport.send(mido.Message('sysex', data=msg))

def wait_for_response(timeout=1.0):
    start = time.time()
    while time.time() - start < timeout:
        msg = inport.poll()
        if msg and msg.type == 'sysex' and len(msg.data) > 10:
            data = msg.data
            # Verifica se è un DT1 di risposta (0x12)
            if data[5] == 0x12:
                addr = data[6:10]
                val = data[10]
                print(f"📥 RX:  {' '.join(f'{b:02X}' for b in data)}")
                return list(addr), val
        time.sleep(0.01)
    print("❌ Nessuna risposta")
    return None, None

## Test Lettura Singoli Parametri

In [ ]:
# Mappa indirizzi nota
ADDR_MAP = {
    "MODEL": [0x20, 0x00, 0x00, 0x06],
    "GAIN":  [0x20, 0x00, 0x00, 0x04],
    "BASS":  [0x20, 0x00, 0x00, 0x00]
}

with mido.open_output(IR2_OUTPUT) as outport:
    with mido.open_input(IR2_INPUT) as inport:
        
        for name, addr in ADDR_MAP.items():
            print(f"\nLeggo {name}...")
            send_rq1(addr)
            ret_addr, val = wait_for_response()
            
            if val is not None:
                print(f"✅ {name} = {val}")